In [35]:
import requests
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [36]:
# GET DATA
api_key = "75f28d5d0dc3441689f03290985d0a97"
keyword = "covid"

URL = f"https://newsapi.org/v2/everything?q={keyword}&sortBy=publishedAt&pageSize=100&language=en&apiKey={api_key}"
response = requests.get(URL)
data_json = response.json()

url = "https://srhdpeuwpubsa.blob.core.windows.net/whdh/COVID/WHO-COVID-19-global-daily-data.csv"
df = pd.read_csv(url)

response = requests.head(url)
ukuran_bytes = int(response.headers.get('content-length', 0))

ukuran_mb = ukuran_bytes / (1024 * 1024)
print(f"Ukuran File CSV di URL: {ukuran_mb:.2f} MB")

Ukuran File CSV di URL: 25.32 MB


In [23]:
# Clean null values pada kasus baru
df_clean = df.dropna(subset=['New_cases']).copy()

# FEATURE ENGINEERING (Membuat Label Risk_Level - Supervised Learning)
def assign_risk(cases):
    if cases < 100:
        return 'Rendah'
    elif cases < 1000:
        return 'Sedang'  # <--- Sudah diperbaiki dari 'Seadang'
    else:
        return 'Tinggi'

df_clean['Risk_Level'] = df_clean['New_cases'].apply(assign_risk)

In [25]:
# 1. UNSUPERVISED LEARNING - K-Means Clustering
X_usl = df_clean[['Cumulative_cases', 'Cumulative_deaths']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_clean['Cluster_USL'] = kmeans.fit_predict(X_usl)

# 2. FEATURE ENGINEERING TAMBAHAN (Ekstraksi Waktu & Wilayah)
df_clean['Date_reported'] = pd.to_datetime(df_clean['Date_reported'])
df_clean['Month'] = df_clean['Date_reported'].dt.month
df_clean['DayOfWeek'] = df_clean['Date_reported'].dt.dayofweek

# One-Hot Encoding untuk wilayah (WHO_region)
df_encoded = pd.get_dummies(df_clean, columns=['WHO_region'], drop_first=True)

# 3. PREPARASI DATASET UNTUK SUPERVISED LEARNING (SL)
features = ['Cumulative_cases', 'Cumulative_deaths', 'Cluster_USL', 'Month', 'DayOfWeek'] + \
           [col for col in df_encoded.columns if col.startswith('WHO_region_')]

X = df_encoded[features]
y = df_encoded['Risk_Level']

In [30]:
import sqlite3
import pandas as pd

# ==========================================
# NODE 17: IS DATA GROWTH > 0 ?
# ==========================================
# Cek apakah jumlah baris data terolah bertambah / lebih dari 0
data_growth = len(df_clean)

print(f"--- PENGECEKAN NODE 17 ---")
print(f"Jumlah baris data siap simpan: {data_growth} baris")

if data_growth > 0:
    print("[STATUS: YES] Pertumbuhan data > 0. Lanjut ke Node 18 (Storage).\n")

    # ==========================================
    # NODE 18: STORE DATA BY PERIODIC (DATABASE)
    # ==========================================
    # 1. Membuat/Menghubungkan ke Database SQLite lokal
    conn = sqlite3.connect('covid19_bigdata_storage.db')

    # 2. Menyimpan DataFrame ke dalam tabel database 'covid19_periodic_data'
    # if_exists='replace' akan memperbarui tabel secara berkala
    df_clean.to_sql('covid19_periodic_data', conn, if_exists='replace', index=False)

    print("--- HASIL PENYIMPANAN NODE 18 ---")
    print("[STATUS: SUCCESS] Data berhasil disimpan ke Database 'covid19_bigdata_storage.db'")
    print("Nama Tabel: covid19_periodic_data")

    # 3. Verifikasi Data yang Tersimpan di Database
    query_check = "SELECT Date_reported, Country, New_cases, Cumulative_cases, Risk_Level, Cluster_USL FROM covid19_periodic_data LIMIT 5"
    df_from_db = pd.read_sql_query(query_check, conn)

    # Tutup koneksi database
    conn.close()

    print("\n--- Preview 5 Baris Data Pertama dari Database SQL ---")
    print(df_from_db)

else:
    print("[STATUS: NO] Data kosong! Kembali ke Node 13 (Redesign Architecture).")

--- PENGECEKAN NODE 17 ---
Jumlah baris data siap simpan: 215026 baris
[STATUS: YES] Pertumbuhan data > 0. Lanjut ke Node 18 (Storage).

--- HASIL PENYIMPANAN NODE 18 ---
[STATUS: SUCCESS] Data berhasil disimpan ke Database 'covid19_bigdata_storage.db'
Nama Tabel: covid19_periodic_data

--- Preview 5 Baris Data Pertama dari Database SQL ---
  Date_reported            Country  New_cases  Cumulative_cases Risk_Level  \
0    2020-01-04     American Samoa        0.0                 0     Rendah   
1    2020-01-04             Bhutan        0.0                 0     Rendah   
2    2020-01-04         Bangladesh        0.0                 0     Rendah   
3    2020-01-04  Brunei Darussalam        0.0                 0     Rendah   
4    2020-01-04              China        1.0                 1     Rendah   

   Cluster_USL  
0            0  
1            0  
2            0  
3            0  
4            0  


In [27]:
# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# MACHINE LEARNING METHOD (Random Forest - Hyperparameter Tuned)
# Meningkatkan n_estimators ke 100 dan max_depth ke 10 agar akurasi naik ke rentang 70%-89%
model_sl = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model_sl.fit(X_train, y_train)

# CEK AKURASI MODEL
y_pred = model_sl.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100

print(f"--- Evaluasi Sesuai Flowchart ---")
print(f"Akurasi Model: {acc:.2f}%")

if 70 <= acc < 90:
    print("[STATUS: PAS] Akurasi memenuhi syarat (70% <= x < 90%). Lanjut ke tahap berikutnya!")
else:
    print("[STATUS: REVISI] Akurasi di luar rentang. Sesuaikan hyperparameter model.")

print("\nDetail Laporan Klasifikasi:")
print(classification_report(y_test, y_pred))

--- Evaluasi Sesuai Flowchart ---
Akurasi Model: 75.03%
[STATUS: PAS] Akurasi memenuhi syarat (70% <= x < 90%). Lanjut ke tahap berikutnya!

Detail Laporan Klasifikasi:
              precision    recall  f1-score   support

      Rendah       0.84      0.90      0.87     35973
      Sedang       0.61      0.39      0.48     15273
      Tinggi       0.63      0.77      0.69     13262

    accuracy                           0.75     64508
   macro avg       0.69      0.69      0.68     64508
weighted avg       0.74      0.75      0.74     64508



In [29]:
# CELL 6: COMPARE MODELING (Versi Cepat untuk Data Besar)
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Scaling Fitur (Wajib untuk SGD/SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Latih Model Kedua (SGDClassifier loss='hinge' = Linear SVM versi sangat cepat)
# Ini berjalan dalam 1-3 detik saja meskipun data ratusan ribu baris!
model_svm = SGDClassifier(loss='hinge', max_iter=1000, random_state=42)
model_svm.fit(X_train_scaled, y_train)

# 3. Prediksi & Evaluasi SVM
y_pred_svm = model_svm.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm) * 100

print("--- HASIL PERBANDINGAN MODELING ---")
print(f"1. Random Forest Accuracy : {acc:.2f}%")
print(f"2. Linear SVM (SGD)       : {acc_svm:.2f}%")

if acc > acc_svm:
    print("\n[KESIMPULAN] Random Forest dipilih sebagai BEST MODEL!")
else:
    print("\n[KESIMPULAN] Linear SVM dipilih sebagai BEST MODEL!")

--- HASIL PERBANDINGAN MODELING ---
1. Random Forest Accuracy : 75.03%
2. Linear SVM (SGD)       : 60.74%

[KESIMPULAN] Random Forest dipilih sebagai BEST MODEL!
